# Zero-Shot Classification with EchoCare-CLIP Models

This notebook evaluates the trained EchoCare-CLIP models from `experiment_training_Mar25.ipynb`
on zero-shot classification tasks using the held-out test set.

## Classification Tasks
1. **Tissue / organ classification** — predict the anatomical region from the image
2. **Condition classification** — predict benign / malignant / normal (where labels are available)

## Method
Standard CLIP zero-shot: encode class text prompts into prototypes, encode test images,
classify by maximum cosine similarity.

## Sections
0. Setup
1. Configuration
2. Load test data from saved splits
3. Model definitions & loading
4. Label extraction & prompt engineering
5. Zero-shot classification
6. Evaluation & visualization

---
## Section 0 — Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q monai torch torchvision transformers einops matplotlib scikit-learn tqdm pandas Pillow scipy seaborn

In [ ]:
import os
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from transformers import CLIPTokenizer, CLIPTextModel
from monai.networks.nets.swin_unetr import SwinTransformer

from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, top_k_accuracy_score
)
from tqdm.auto import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

---
## Section 1 — Configuration

Architecture parameters **must** match the training notebook exactly.

In [ ]:
config = {
    # ── Paths ──────────────────────────────────────────────────────────────
    'prepared_data_root': '/content/drive/MyDrive/BMI702 Project/ Ultrasound Data/prepared_data',
    'echocare_checkpoint': '/content/drive/MyDrive/BMI702 Project/echocare_encoder.pth',
    'saved_models_dir': '/content/drive/MyDrive/BMI702 Project/saved_models',

    # ── Image Encoder (EchoCare SwinTransformer) ───────────────────────────
    'feature_size'   : 128,
    'in_channels'    : 3,
    'image_size'     : 256,

    # ── Text Encoder (CLIP) ────────────────────────────────────────────────
    'clip_model_name': 'openai/clip-vit-base-patch32',
    'max_seq_len'    : 77,

    # ── Shared Latent Space ────────────────────────────────────────────────
    'projection_dim' : 256,

    # ── Normalization (ImageNet defaults — override if training used custom stats) ──
    'img_mean': [0.485, 0.456, 0.406],
    'img_std' : [0.229, 0.224, 0.225],

    # ── Training params (needed for model instantiation) ───────────────────
    'init_temperature': 0.07,

    # ── DataLoader ─────────────────────────────────────────────────────────
    'batch_size' : 16,
    'num_workers': 2,
}

# ── Model checkpoint filenames ─────────────────────────────────────────────
MODEL_CHECKPOINTS = {
    'Exp 1: MLP only':        ('echocare_clip_mlp_heads_only.pt',        True,  True),
    'Exp 2: MLP + Img Enc':   ('echocare_clip_mlp_and_image_encoder.pt', False, True),
    'Exp 3: MLP + Txt Enc':   ('echocare_clip_mlp_and_text_encoder.pt',  True,  False),
}
# Format: (filename, freeze_image, freeze_text) — must match training

print('Configuration loaded.')
for k, v in config.items():
    print(f'  {k:30s}: {v}')

---
## Section 2 — Load Test Data from Saved Splits

In [ ]:
splits_dir = os.path.join(config['prepared_data_root'], 'data_splits')

# ── Load metadata ──────────────────────────────────────────────────────────
meta_path = os.path.join(splits_dir, 'split_metadata.json')
with open(meta_path) as f:
    split_metadata = json.load(f)

print('Split metadata loaded:')
print(f'  Created:    {split_metadata["timestamp"]}')
print(f'  Columns:    {split_metadata["full_df_columns"]}')
print(f'  Split sizes: {split_metadata["split_sizes"]}')
print(f'  Has tissue column:    {split_metadata["has_tissue_column"]}')
print(f'  Has condition column: {split_metadata["has_condition_column"]}')
print(f'  Library versions: {split_metadata["library_versions"]}')

# ── Load test DataFrame ───────────────────────────────────────────────────
test_df = pd.read_parquet(os.path.join(splits_dir, 'test_df.parquet'))
print(f'\nTest set: {len(test_df):,} samples, columns={list(test_df.columns)}')
print(f'Datasets represented: {sorted(test_df["dataset"].unique())}')
print(test_df.head())

In [ ]:
class MultiDatasetUltrasound(Dataset):
    """
    Multi-source ultrasound dataset with free-text captions.

    Returns per sample:
      image   : (3, image_size, image_size) float tensor, normalized
      caption : str  — clinical text description
      dataset : str  — source dataset name
    """

    def __init__(self, dataframe: pd.DataFrame, split: str = 'test', cfg: dict = config):
        self.df    = dataframe.reset_index(drop=True)
        self.split = split

        sz   = cfg['image_size']
        mean = cfg['img_mean']
        std  = cfg['img_std']

        if split == 'train':
            self.transform = transforms.Compose([
                transforms.Resize(int(sz * 1.125)),
                transforms.RandomCrop(sz),
                transforms.RandomHorizontalFlip(),
                transforms.ToTensor(),
                transforms.Normalize(mean=mean, std=std),
            ])
        else:
            self.transform = transforms.Compose([
                transforms.Resize(sz),
                transforms.CenterCrop(sz),
                transforms.ToTensor(),
                transforms.Normalize(mean=mean, std=std),
            ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        image = Image.open(row['image_path']).convert('RGB')
        image = self.transform(image)
        return image, row['caption'], row['dataset']


test_dataset = MultiDatasetUltrasound(test_df, split='test')
test_loader  = DataLoader(
    test_dataset,
    batch_size=config['batch_size'],
    shuffle=False,
    drop_last=False,
    num_workers=config['num_workers'],
    pin_memory=True,
)
print(f'Test DataLoader: {len(test_loader)} batches ({len(test_dataset):,} samples)')

# Sanity check
img, cap, src = test_dataset[0]
print(f'Sample — image={tuple(img.shape)}, dataset={src!r}, caption[:80]={cap[:80]!r}')

---
## Section 3 — Model Definitions & Loading

Model architectures are copied from the training notebook to keep this notebook self-contained.

In [ ]:
class EchoCareImageEncoder(nn.Module):
    """
    EchoCare SwinTransformer backbone + MLP projection head.

    Input  : (B, 3, 256, 256)
    Output : (B, projection_dim)  L2-normalized
    """

    def __init__(
        self,
        feature_size   : int  = 128,
        in_channels    : int  = 3,
        projection_dim : int  = 256,
        use_checkpoint : bool = True,
        freeze_encoder : bool = False,
    ):
        super().__init__()

        self.encoder = SwinTransformer(
            in_chans       = in_channels,
            embed_dim      = feature_size,
            window_size    = [8, 8],
            patch_size     = [2, 2],
            depths         = [2, 2, 18, 2],
            num_heads      = [4, 8, 16, 32],
            mlp_ratio      = 4.0,
            qkv_bias       = True,
            use_checkpoint = use_checkpoint,
            spatial_dims   = 2,
            use_v2         = True,
        )

        encoder_out_dim = feature_size * (2 ** 4)  # 128 * 16 = 2048

        self.projection = nn.Sequential(
            nn.Linear(encoder_out_dim, encoder_out_dim),
            nn.ReLU(),
            nn.Linear(encoder_out_dim, projection_dim),
        )

        if freeze_encoder:
            for param in self.encoder.parameters():
                param.requires_grad = False

    def load_pretrained(self, checkpoint_path: str, device: str = 'cpu'):
        """Load EchoCare MAE pretrained weights (removes mask_token artifact)."""
        state_dict = torch.load(checkpoint_path, map_location=device)
        state_dict.pop('mask_token', None)
        self.encoder.load_state_dict(state_dict, strict=True)
        print(f"Loaded EchoCare pretrained weights from '{checkpoint_path}'")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feats = self.encoder(x)           # list of feature maps
        deep  = feats[-1]                 # (B, 2048, 8, 8)
        emb   = deep.mean(dim=(2, 3))     # GAP -> (B, 2048)
        emb   = self.projection(emb)      # (B, projection_dim)
        return F.normalize(emb, dim=-1)


class CLIPTextEncoder(nn.Module):
    """
    CLIP text tower + MLP projection head.

    Input  : (B, 77) token ids
    Output : (B, projection_dim)  L2-normalized
    """

    def __init__(
        self,
        clip_model_name     : str  = 'openai/clip-vit-base-patch32',
        projection_dim      : int  = 256,
        freeze_text_encoder : bool = True,
    ):
        super().__init__()

        self.text_encoder = CLIPTextModel.from_pretrained(clip_model_name)
        text_out_dim      = self.text_encoder.config.hidden_size  # 512

        if freeze_text_encoder:
            for param in self.text_encoder.parameters():
                param.requires_grad = False

        self.projection = nn.Sequential(
            nn.Linear(text_out_dim, text_out_dim),
            nn.ReLU(),
            nn.Linear(text_out_dim, projection_dim),
        )

    def forward(
        self,
        input_ids     : torch.Tensor,
        attention_mask: torch.Tensor | None = None,
    ) -> torch.Tensor:
        outputs  = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        text_emb = outputs.pooler_output          # (B, 512)
        text_emb = self.projection(text_emb)      # (B, projection_dim)
        return F.normalize(text_emb, dim=-1)


class EchoCare_CLIP(nn.Module):
    """
    CLIP-style contrastive model combining EchoCare image encoder and CLIP text encoder.
    """

    def __init__(
        self,
        image_encoder   : EchoCareImageEncoder,
        text_encoder    : CLIPTextEncoder,
        init_temperature: float = 0.07,
    ):
        super().__init__()
        self.image_encoder   = image_encoder
        self.text_encoder    = text_encoder
        self.log_temperature = nn.Parameter(torch.log(torch.tensor(init_temperature)))

    def encode_image(self, images: torch.Tensor) -> torch.Tensor:
        return self.image_encoder(images)

    def encode_text(self, input_ids, attention_mask=None) -> torch.Tensor:
        return self.text_encoder(input_ids, attention_mask)

    def forward(self, images, input_ids, attention_mask=None):
        image_emb = self.encode_image(images)
        text_emb  = self.encode_text(input_ids, attention_mask)

        temperature      = self.log_temperature.exp()
        logits_per_image = (image_emb @ text_emb.T) / temperature
        logits_per_text  = logits_per_image.T

        labels   = torch.arange(image_emb.size(0), device=image_emb.device)
        loss_i2t = F.cross_entropy(logits_per_image, labels)
        loss_t2i = F.cross_entropy(logits_per_text,  labels)
        loss     = (loss_i2t + loss_t2i) / 2

        return loss, image_emb, text_emb


def tokenize_texts(texts, tokenizer, max_length=77, device='cpu'):
    """Tokenize a list of strings with the CLIP tokenizer."""
    encoded = tokenizer(
        texts,
        padding        = 'max_length',
        max_length     = max_length,
        truncation     = True,
        return_tensors = 'pt',
    )
    return (
        encoded['input_ids'].to(device),
        encoded['attention_mask'].to(device),
    )


tokenizer = CLIPTokenizer.from_pretrained(config['clip_model_name'])
print(f'Model classes and tokenizer loaded  |  vocab_size = {tokenizer.vocab_size:,}')

In [ ]:
def build_model(freeze_image=True, freeze_text=True):
    """
    Instantiate EchoCare_CLIP, load pretrained EchoCare image encoder weights,
    and apply freeze flags for each encoder.
    """
    img_enc = EchoCareImageEncoder(
        feature_size   = config['feature_size'],
        in_channels    = config['in_channels'],
        projection_dim = config['projection_dim'],
        freeze_encoder = freeze_image,
    )
    if os.path.exists(config['echocare_checkpoint']):
        img_enc.load_pretrained(config['echocare_checkpoint'], device=str(device))
    else:
        print('[WARNING] EchoCare checkpoint not found — random image encoder weights')

    txt_enc = CLIPTextEncoder(
        clip_model_name     = config['clip_model_name'],
        projection_dim      = config['projection_dim'],
        freeze_text_encoder = freeze_text,
    )

    model = EchoCare_CLIP(
        image_encoder    = img_enc,
        text_encoder     = txt_enc,
        init_temperature = config['init_temperature'],
    ).to(device)

    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in model.parameters())
    print(f'  Trainable: {n_train:,} / {n_total:,}  ({100*n_train/n_total:.1f}%)')
    return model


print('build_model() defined.')

In [ ]:
# ── Load all trained models ────────────────────────────────────────────────
models = {}

for exp_name, (ckpt_file, freeze_img, freeze_txt) in MODEL_CHECKPOINTS.items():
    print(f'\nLoading {exp_name}...')
    ckpt_path = os.path.join(config['saved_models_dir'], ckpt_file)
    model = build_model(freeze_image=freeze_img, freeze_text=freeze_txt)
    state_dict = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(state_dict)
    model.eval()
    models[exp_name] = model
    print(f'  Loaded from {ckpt_path}')

print(f'\nAll {len(models)} models loaded successfully.')

In [ ]:
# ── Sanity check: forward pass through each model ──────────────────────────
sample_imgs, sample_caps, _ = next(iter(test_loader))
sample_imgs = sample_imgs.to(device)
sample_ids, sample_mask = tokenize_texts(list(sample_caps), tokenizer, device=device)

for name, model in models.items():
    with torch.no_grad():
        img_emb = model.encode_image(sample_imgs)
        txt_emb = model.encode_text(sample_ids, sample_mask)
    print(f'{name:25s} | img_emb={tuple(img_emb.shape)} | txt_emb={tuple(txt_emb.shape)}')

print('\nAll models produce correct output shapes.')

---
## Section 4 — Label Extraction & Prompt Engineering

Extract ground-truth labels from metadata columns (if available) or by parsing captions.
Build text prompt ensembles for each class using the same prompt templates from training.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Strategy: use metadata columns if available, otherwise parse from captions
# ══════════════════════════════════════════════════════════════════════════════

# ── Tissue label extraction ────────────────────────────────────────────────
if 'tissue' in test_df.columns:
    print('Using metadata column for tissue labels.')
    test_df['tissue_label'] = test_df['tissue'].str.strip().str.lower()
else:
    print('No tissue metadata column found — parsing from captions...')
    # Extract tissue from prompt template patterns:
    #   "ultrasound image of {Tissue}", "B-mode ultrasound showing {Tissue}",
    #   "Sonographic appearance of {Tissue}", etc.
    TISSUE_PATTERNS = [
        r'ultrasound (?:image |scan )?of ([^.,;]+?)(?:\.|,|;|$)',
        r'ultrasound showing ([^.,;]+?)(?:\.|,|;|$)',
        r'[Ss]onographic appearance of ([^.,;]+?)(?:\.|,|;|$)',
        r'[Ee]chographic image of ([^.,;]+?)(?:\.|,|;|$)',
        r'[Ss]onogram shows ([^.,;]+?)(?:\.|,|;|$)',
        r'[Ee]xamination of ([^.,;]+?)(?:\s+reveal|\.|,|;|$)',
    ]

    def extract_tissue_from_caption(caption):
        """Extract tissue/organ name from caption using prompt template patterns."""
        for pattern in TISSUE_PATTERNS:
            m = re.search(pattern, caption, re.IGNORECASE)
            if m:
                tissue = m.group(1).strip().lower()
                # Remove common modifiers to normalize
                tissue = re.sub(r'\b(the|a|an)\b', '', tissue).strip()
                # Remove trailing condition words if they leaked in
                tissue = re.sub(r'\s+with\s+.*$', '', tissue).strip()
                if tissue:
                    return tissue
        return None

    test_df['tissue_label'] = test_df['caption'].apply(extract_tissue_from_caption)

# ── Condition label extraction ─────────────────────────────────────────────
if 'condition' in test_df.columns:
    print('Using metadata column for condition labels.')
    test_df['condition_label'] = test_df['condition'].str.strip().str.lower()
else:
    print('No condition metadata column found — parsing from captions...')
    CONDITION_KEYWORDS = {
        'benign':    [r'\bbenign\b'],
        'malignant': [r'\bmalignant\b', r'\bcancer\b', r'\bcarcinoma\b'],
        'normal':    [r'\bnormal\b', r'\bhealthy\b', r'\bunremarkable\b'],
    }

    def extract_condition_from_caption(caption):
        """Extract condition label from caption using keyword matching."""
        cap_lower = caption.lower()
        for label, patterns in CONDITION_KEYWORDS.items():
            for p in patterns:
                if re.search(p, cap_lower):
                    return label
        return None

    test_df['condition_label'] = test_df['caption'].apply(extract_condition_from_caption)

# ── Summary ────────────────────────────────────────────────────────────────
n_tissue = test_df['tissue_label'].notna().sum()
n_cond   = test_df['condition_label'].notna().sum()
print(f'\nLabel extraction summary:')
print(f'  Tissue labels:    {n_tissue:,} / {len(test_df):,} ({100*n_tissue/len(test_df):.1f}%)')
print(f'  Condition labels: {n_cond:,} / {len(test_df):,} ({100*n_cond/len(test_df):.1f}%)')

if n_tissue > 0:
    print(f'\nTissue label distribution:')
    print(test_df['tissue_label'].value_counts().to_string())
if n_cond > 0:
    print(f'\nCondition label distribution:')
    print(test_df['condition_label'].value_counts().to_string())

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Build text prompt ensembles for each class
# Uses the same prompt templates from prompt_template_explanations.md
# ══════════════════════════════════════════════════════════════════════════════

# ── Tier 1 templates: tissue only ──────────────────────────────────────────
TIER1_TEMPLATES = [
    "An ultrasound image of {}.",
    "A B-mode ultrasound showing {}.",
    "Sonographic appearance of {}.",
    "This is an ultrasound image of {}.",
    "A grayscale ultrasound image demonstrating {}.",
    "Ultrasound findings consistent with {}.",
    "A clinical ultrasound scan of {}.",
    "An echographic image of {}.",
    "This sonogram shows {}.",
    "A diagnostic ultrasound image of {}, obtained for clinical evaluation.",
]

# ── Tier 2 templates: tissue + condition ───────────────────────────────────
TIER2_TEMPLATES = [
    "An ultrasound image of {} with {} findings.",
    "A B-mode ultrasound of {}, consistent with {}.",
    "Sonographic appearance of {} {}.",
    "This ultrasound demonstrates {} exhibiting features of {}.",
    "A clinical ultrasound scan of {}, indicative of {} pathology.",
    "Echographic findings of {} showing {} characteristics.",
    "This sonogram of {} is consistent with a {} diagnosis.",
    "A diagnostic ultrasound image of {}, with imaging features suggestive of {}.",
    "Ultrasound of {} presenting sonographic signs of {}.",
    "A grayscale ultrasound demonstrating {} changes in {}.",
]


def build_tissue_prompts(class_names):
    """Build a dict mapping class_name -> list of 10 prompt strings."""
    return {name: [t.format(name) for t in TIER1_TEMPLATES] for name in class_names}


def build_condition_prompts(tissue_names, condition_names):
    """
    Build prompts for condition classification.
    For each condition, generate prompts across all tissue types to make
    the classifier tissue-agnostic (average across tissues).
    """
    prompts = {}
    for cond in condition_names:
        cond_prompts = []
        for tissue in tissue_names:
            cond_prompts.extend([t.format(tissue, cond) for t in TIER2_TEMPLATES])
        prompts[cond] = cond_prompts
    return prompts


# ── Build prompt sets for each classification task ─────────────────────────
classification_tasks = {}

# Task 1: Tissue classification
tissue_mask = test_df['tissue_label'].notna()
if tissue_mask.sum() > 0:
    tissue_classes = sorted(test_df.loc[tissue_mask, 'tissue_label'].unique())
    tissue_prompts = build_tissue_prompts(tissue_classes)
    classification_tasks['Tissue'] = {
        'class_names': tissue_classes,
        'prompts':     tissue_prompts,
        'mask':        tissue_mask,
        'labels':      test_df.loc[tissue_mask, 'tissue_label'].values,
    }
    print(f'Tissue classification: {len(tissue_classes)} classes, '
          f'{tissue_mask.sum():,} test samples')
    for c in tissue_classes:
        print(f'  {c}: {(test_df.loc[tissue_mask, "tissue_label"] == c).sum()} samples, '
              f'{len(tissue_prompts[c])} prompts')

# Task 2: Condition classification
cond_mask = test_df['condition_label'].notna()
if cond_mask.sum() > 0:
    cond_classes = sorted(test_df.loc[cond_mask, 'condition_label'].unique())
    # Use the tissue classes (or dataset names as proxy) for condition prompt generation
    tissues_for_cond = tissue_classes if 'Tissue' in classification_tasks else ['tissue']
    cond_prompts = build_condition_prompts(tissues_for_cond, cond_classes)
    classification_tasks['Condition'] = {
        'class_names': cond_classes,
        'prompts':     cond_prompts,
        'mask':        cond_mask,
        'labels':      test_df.loc[cond_mask, 'condition_label'].values,
    }
    print(f'\nCondition classification: {len(cond_classes)} classes, '
          f'{cond_mask.sum():,} test samples')
    for c in cond_classes:
        print(f'  {c}: {(test_df.loc[cond_mask, "condition_label"] == c).sum()} samples, '
              f'{len(cond_prompts[c])} prompts')

if not classification_tasks:
    raise RuntimeError(
        'No classification labels could be extracted! '
        'Check that captions contain parseable tissue or condition information.'
    )

print(f'\nTotal classification tasks: {list(classification_tasks.keys())}')

---
## Section 5 — Zero-Shot Classification

For each model and each classification task:
1. Encode all class prompts with the text encoder, average per class to get prototypes
2. Encode test images with the image encoder
3. Predict by argmax cosine similarity to prototypes

In [ ]:
@torch.no_grad()
def build_class_prototypes(model, class_names, class_prompts, tokenizer, device):
    """
    Encode all class prompts and average per class to get prototypes.
    Returns: (K, D) tensor of L2-normalized class prototypes on CPU.
    """
    prototypes = []
    for cls_name in class_names:
        prompts = class_prompts[cls_name]
        ids, mask = tokenize_texts(prompts, tokenizer, device=device)
        text_embs = model.encode_text(ids, mask)  # (num_prompts, D)
        prototype = text_embs.mean(dim=0)          # (D,)
        prototype = F.normalize(prototype, dim=0)
        prototypes.append(prototype)
    return torch.stack(prototypes).cpu()  # (K, D) — move to CPU to match image embeddings


@torch.no_grad()
def extract_image_embeddings(model, dataloader, device):
    """
    Encode all images in the dataloader.
    Returns: (N, D) tensor of L2-normalized image embeddings.
    """
    model.eval()
    all_embs = []
    for images, _, _ in tqdm(dataloader, desc='Encoding images', leave=False):
        images = images.to(device)
        embs = model.encode_image(images)
        all_embs.append(embs.cpu())
    return torch.cat(all_embs)  # (N, D)


def zero_shot_classify(img_embs, class_prototypes):
    """
    Classify images by cosine similarity to class prototypes.
    Both inputs are L2-normalized, so dot product = cosine similarity.

    Args:
        img_embs:         (N, D) image embeddings
        class_prototypes: (K, D) class prototypes

    Returns:
        predictions: (N,) predicted class indices
        similarities: (N, K) cosine similarity matrix
    """
    similarities = img_embs @ class_prototypes.T  # (N, K)
    predictions  = similarities.argmax(dim=1).numpy()
    return predictions, similarities.numpy()


print('Zero-shot classification functions defined.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Run zero-shot classification for all models x all tasks
# ══════════════════════════════════════════════════════════════════════════════

all_results = {}  # (model_name, task_name) -> dict of results

for model_name, model in models.items():
    print(f'\n{"="*60}')
    print(f'  {model_name}')
    print(f'{"="*60}')

    # Extract image embeddings once per model (shared across tasks)
    img_embs_full = extract_image_embeddings(model, test_loader, device)
    print(f'  Image embeddings: {tuple(img_embs_full.shape)}')

    for task_name, task_info in classification_tasks.items():
        print(f'\n  --- {task_name} classification ---')

        class_names = task_info['class_names']
        gt_labels   = task_info['labels']
        mask        = task_info['mask']

        # Build class prototypes
        prototypes = build_class_prototypes(
            model, class_names, task_info['prompts'], tokenizer, device
        )

        # Select image embeddings for samples that have labels
        mask_indices = np.where(mask.values)[0]
        img_embs = img_embs_full[mask_indices]

        # Classify
        pred_indices, similarities = zero_shot_classify(img_embs, prototypes)
        pred_labels = np.array([class_names[i] for i in pred_indices])

        # Compute metrics
        acc = accuracy_score(gt_labels, pred_labels)
        f1_macro = f1_score(gt_labels, pred_labels, average='macro', zero_division=0)
        f1_weighted = f1_score(gt_labels, pred_labels, average='weighted', zero_division=0)

        print(f'  Accuracy:     {acc:.4f}')
        print(f'  F1 (macro):   {f1_macro:.4f}')
        print(f'  F1 (weighted): {f1_weighted:.4f}')

        all_results[(model_name, task_name)] = {
            'class_names':  class_names,
            'gt_labels':    gt_labels,
            'pred_labels':  pred_labels,
            'pred_indices': pred_indices,
            'similarities': similarities,
            'accuracy':     acc,
            'f1_macro':     f1_macro,
            'f1_weighted':  f1_weighted,
        }

print(f'\nDone. {len(all_results)} (model, task) evaluations completed.')

---
## Section 6 — Evaluation & Visualization

In [ ]:
# ── Per-class classification reports ───────────────────────────────────────
for (model_name, task_name), res in all_results.items():
    print(f'\n{"="*60}')
    print(f'  {model_name} — {task_name} Classification')
    print(f'{"="*60}')
    print(classification_report(
        res['gt_labels'], res['pred_labels'],
        labels=res['class_names'],
        zero_division=0,
    ))

In [ ]:
# ── Confusion matrices ────────────────────────────────────────────────────
task_names = list(classification_tasks.keys())
model_names = list(models.keys())
n_tasks  = len(task_names)
n_models = len(model_names)

fig, axes = plt.subplots(
    n_tasks, n_models,
    figsize=(6 * n_models, 5 * n_tasks),
    squeeze=False,
)

for row, task_name in enumerate(task_names):
    for col, model_name in enumerate(model_names):
        ax  = axes[row, col]
        key = (model_name, task_name)

        if key not in all_results:
            ax.set_visible(False)
            continue

        res = all_results[key]
        cm  = confusion_matrix(
            res['gt_labels'], res['pred_labels'],
            labels=res['class_names'],
        )
        sns.heatmap(
            cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=res['class_names'],
            yticklabels=res['class_names'],
            ax=ax,
        )
        ax.set_xlabel('Predicted')
        ax.set_ylabel('True')
        ax.set_title(f'{model_name}\n{task_name} (acc={res["accuracy"]:.3f})')

plt.tight_layout()
plt.show()

In [ ]:
# ── Summary comparison: accuracy and F1 across models ──────────────────────
summary_rows = []
for (model_name, task_name), res in all_results.items():
    summary_rows.append({
        'Model': model_name,
        'Task':  task_name,
        'Accuracy': res['accuracy'],
        'F1 (macro)': res['f1_macro'],
        'F1 (weighted)': res['f1_weighted'],
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

# ── Bar chart ──────────────────────────────────────────────────────────────
metrics = ['Accuracy', 'F1 (macro)', 'F1 (weighted)']
fig, axes = plt.subplots(1, len(task_names), figsize=(7 * len(task_names), 5))
if len(task_names) == 1:
    axes = [axes]

bar_colors = ['#4C72B0', '#DD8452', '#55A868']

for ax, task_name in zip(axes, task_names):
    task_df = summary_df[summary_df['Task'] == task_name]
    x = np.arange(len(task_df))
    width = 0.25

    for i, metric in enumerate(metrics):
        ax.bar(x + i * width, task_df[metric].values, width,
               label=metric, color=bar_colors[i], edgecolor='white')

    ax.set_xlabel('Model')
    ax.set_ylabel('Score')
    ax.set_title(f'{task_name} Classification')
    ax.set_xticks(x + width)
    ax.set_xticklabels(task_df['Model'].values, rotation=15, ha='right')
    ax.set_ylim(0, 1)
    ax.legend(loc='lower right')
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── Top-k accuracy (if enough classes) ────────────────────────────────────
for (model_name, task_name), res in all_results.items():
    n_classes = len(res['class_names'])
    if n_classes < 3:
        continue

    # Convert gt labels to indices
    label_to_idx = {name: i for i, name in enumerate(res['class_names'])}
    gt_indices = np.array([label_to_idx[l] for l in res['gt_labels']])

    for k in [3, 5]:
        if k >= n_classes:
            continue
        topk_acc = top_k_accuracy_score(
            gt_indices, res['similarities'],
            k=k, labels=np.arange(n_classes)
        )
        print(f'{model_name:25s} | {task_name:12s} | Top-{k} accuracy: {topk_acc:.4f}')